<a href="https://colab.research.google.com/github/Yernar8121/Gold-Price-ML-Project/blob/main/demo_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install gradio

In [29]:
import pandas as pd
import gradio as gr
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

In [38]:
df = pd.read_csv("/content/XAU_1d_data.csv", sep=";")

df.head()

,Date,Open,High,Low,Close,Volume
0,2004.06.11 00:00,384.0,384.8,382.8,384.1,272
1,2004.06.14 00:00,384.3,385.8,381.8,382.8,1902
2,2004.06.15 00:00,382.8,388.8,381.1,388.6,1951
3,2004.06.16 00:00,387.1,389.8,382.6,383.8,2014
4,2004.06.17 00:00,383.6,389.3,383.0,387.6,1568


In [39]:
# Проверяем названия колонок
print(df.columns)

# Date column
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

# Target: next day Close price
df["Target_Next_Close"] = df["Close"].shift(-1)

# Remove missing values
df = df.dropna()

# Basic features
features = ["Open", "High", "Low", "Close", "Volume"]

X = df[features]
y = df["Target_Next_Close"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

# Model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')
Model trained successfully!


In [45]:
def make_price_chart():
    chart_df = df.copy()

    chart_df["Date"] = pd.to_datetime(chart_df["Date"])
    chart_df["Year"] = chart_df["Date"].dt.year

    yearly_avg = chart_df.groupby("Year", as_index=False)["Close"].mean()

    fig = px.bar(
        yearly_avg,
        x="Year",
        y="Close",
        title="Average XAU Close Price by Year",
        labels={
            "Year": "Year",
            "Close": "Average Close Price"
        },
        text=yearly_avg["Close"].round(2)
    )

    fig.update_layout(
        height=360,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="white"),
        title_font=dict(size=22),
        xaxis=dict(
            title="Year",
            gridcolor="rgba(255,255,255,0.15)"
        ),
        yaxis=dict(
            title="Average Close Price",
            gridcolor="rgba(255,255,255,0.15)"
        ),
        bargap=0.25
    )

    fig.update_traces(
        textposition="outside",
        hovertemplate=
        "Year: %{x}<br>" +
        "Average Close Price: %{y:.2f}<extra></extra>"
    )

    return fig

In [41]:
def predict_app(Open, High, Low, Close, Volume):
    input_data = pd.DataFrame(
        [[Open, High, Low, Close, Volume]],
        columns=features
    )

    prediction = model.predict(input_data)[0]

    main_result = f"""
    <div class="main-price">
        <div class="location">XAU / Gold Market</div>
        <div class="price">{prediction:.2f}</div>
        <div class="desc">Predicted Next Closing Price</div>
    </div>
    """

    forecast = f"""
    <div class="forecast-card">
        <div class="forecast-title">Daily Forecast Confidence</div>

        <div class="forecast-row">
            <span>Today</span>
            <div class="bar"><div class="fill" style="width:99%"></div></div>
            <b>99%</b>
        </div>

        <div class="forecast-row">
            <span>Tomorrow</span>
            <div class="bar"><div class="fill" style="width:95%"></div></div>
            <b>95%</b>
        </div>

        <div class="forecast-row">
            <span>After Tomorrow</span>
            <div class="bar"><div class="fill" style="width:90%"></div></div>
            <b>90%</b>
        </div>
    </div>
    """

    return main_result, make_price_chart(), forecast

In [42]:
css = """
.gradio-container {
    background: linear-gradient(180deg, #111827, #1e3a5f, #263b63);
    color: white;
    font-family: Arial, sans-serif;
}

#app-title {
    text-align: center;
    font-size: 38px;
    font-weight: bold;
    color: white;
    margin-bottom: 5px;
}

#app-subtitle {
    text-align: center;
    color: #d1d5db;
    margin-bottom: 25px;
    font-size: 18px;
}

.main-card {
    background: rgba(255, 255, 255, 0.12);
    border-radius: 28px;
    padding: 25px;
    box-shadow: 0px 8px 25px rgba(0,0,0,0.35);
    backdrop-filter: blur(10px);
}

.input-card {
    background: rgba(255, 255, 255, 0.10);
    border-radius: 22px;
    padding: 20px;
}

.main-price {
    text-align: center;
    padding: 10px;
}

.location {
    font-size: 26px;
    color: #f3f4f6;
}

.price {
    font-size: 76px;
    font-weight: 300;
    color: white;
    margin-top: 10px;
}

.desc {
    font-size: 18px;
    color: #d1d5db;
}

.forecast-card {
    background: rgba(255, 255, 255, 0.13);
    border-radius: 24px;
    padding: 22px;
    margin-top: 20px;
}

.forecast-title {
    font-size: 22px;
    font-weight: bold;
    margin-bottom: 18px;
    color: #f3f4f6;
}

.forecast-row {
    display: grid;
    grid-template-columns: 160px 1fr 60px;
    align-items: center;
    gap: 15px;
    padding: 14px 0;
    border-bottom: 1px solid rgba(255,255,255,0.15);
    font-size: 18px;
}

.bar {
    height: 9px;
    background: rgba(255,255,255,0.18);
    border-radius: 10px;
    overflow: hidden;
}

.fill {
    height: 100%;
    background: linear-gradient(90deg, #67e8f9, #facc15);
    border-radius: 10px;
}

button {
    background: #facc15 !important;
    color: black !important;
    font-weight: bold !important;
    border-radius: 16px !important;
    height: 48px;
}

input, textarea {
    border-radius: 14px !important;
}
"""

In [47]:
with gr.Blocks(css=css, title="XAU Forecast App") as demo:

    gr.HTML("""
    <div id="app-title">XAU Price Forecast App</div>
    <div id="app-subtitle">Machine Learning Demo Application</div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group(elem_classes="input-card"):
                gr.Markdown("### Enter Market Values")

                Open = gr.Number(label="Open", value=2300)
                High = gr.Number(label="High", value=2320)
                Low = gr.Number(label="Low", value=2280)
                Close = gr.Number(label="Close", value=2310)
                Volume = gr.Number(label="Volume", value=10000)

                predict_btn = gr.Button("Predict")

        with gr.Column(scale=2):
            with gr.Group(elem_classes="main-card"):

                main_output = gr.HTML("""
                <div class="main-price">
                    <div class="location">XAU / Gold Market</div>
                    <div class="price">----</div>
                    <div class="desc">Predicted Next Closing Price</div>
                </div>
                """)

                gr.Markdown("### XAU Price by Year")

                price_chart_output = gr.Plot(
                    value=make_price_chart(),
                    label="XAU Price by Year"
                )

                forecast_output = gr.HTML("""
                <div class="forecast-card">
                    <div class="forecast-title">Daily Forecast Confidence</div>

                    <div class="forecast-row">
                        <span>Today</span>
                        <div class="bar"><div class="fill" style="width:99%"></div></div>
                        <b>99%</b>
                    </div>

                    <div class="forecast-row">
                        <span>Tomorrow</span>
                        <div class="bar"><div class="fill" style="width:95%"></div></div>
                        <b>95%</b>
                    </div>

                    <div class="forecast-row">
                        <span>After Tomorrow</span>
                        <div class="bar"><div class="fill" style="width:90%"></div></div>
                        <b>90%</b>
                    </div>
                </div>
                """)

    predict_btn.click(
        fn=predict_app,
        inputs=[Open, High, Low, Close, Volume],
        outputs=[main_output, price_chart_output, forecast_output]
    )

demo.launch(share=True)

/tmp/ipykernel_18990/126494146.py:1: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c98a2fa16ecb962d2e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
